In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import pandas as pd
import shutil

# 1. Define Paths
source_folder = '/content/drive/MyDrive/CPTAC-BRCA/LUAD'
output_base_folder = '/content/drive/MyDrive/CPTAC-LUAD'

tumor_folder = '/content/drive/MyDrive/CPTAC-LUAD/TUMOR'
normal_folder = '/content/drive/MyDrive/CPTAC-LUAD/NORMAL'

csv_path = '/content/drive/MyDrive/CPTAC-LUAD/CPTAC_Data.csv'

print("Loading TCIA Master Table...")
try:
    # 2. Read the CSV
    df = pd.read_csv(csv_path)

    # Clean the column to avoid spacing errors, then filter
    df['Tumor'] = df['Tumor'].astype(str).str.strip()
    df_luad = df[df['Tumor'] == 'LUAD']

    print(f"✅ Dropped other cancers. Using {len(df_luad)} LUAD-specific records (out of {len(df)} total).")

    # 3. Create the dictionary using ONLY the filtered LUAD data
    master_dict = dict(zip(
        df_luad['Slide_ID'].astype(str).str.strip(),
        df_luad['Specimen_Type'].astype(str).str.strip()
    ))

    print(f"✅ Loaded {len(master_dict)} slide labels from the table.\n")

    # 4. Get all remaining .svs files that failed the API check
    unsorted_files = glob.glob(os.path.join(source_folder, '*.svs'))

    if not unsorted_files:
        print("🎉 No more files left to sort in the LUAD folder!")
    else:
        print(f"🧹 Found {len(unsorted_files)} files left behind. Cleaning up...\n")

        moved_tumor = 0
        moved_normal = 0
        still_unknown = 0

        for file_path in unsorted_files:
            filename = os.path.basename(file_path)
            slide_id = filename.replace('.svs', '')

            # Look up the slide in our new dictionary
            label = str(master_dict.get(slide_id, "Unknown")).lower()

            if "tumor" in label:
                shutil.move(file_path, os.path.join(tumor_folder, filename))
                print(f"✅ Moved {filename} ➡️ TUMOR")
                moved_tumor += 1
            elif "normal" in label:
                shutil.move(file_path, os.path.join(normal_folder, filename))
                print(f"✅ Moved {filename} ➡️ NORMAL")
                moved_normal += 1
            else:
                print(f"⚠️ Unknown in CSV: {filename}")
                still_unknown += 1

        print("\n🎉 CLEANUP COMPLETE!")
        print(f"Tumor: {moved_tumor}")
        print(f"Normal: {moved_normal}")
        if still_unknown > 0:
            print(f"Still stuck: {still_unknown}")

except FileNotFoundError:
    print(f"❌ Error: Could not find the CSV file at {csv_path}")
    print("Check the file name and make sure it is uploaded to your Drive!")
except KeyError as e:
    print(f"❌ Error: Could not find the expected column {e} in your CSV.")
    print("Open the CSV and make sure the headers are exactly 'Slide_ID' and 'Specimen Type'.")

Loading TCIA Master Table...
✅ Dropped other cancers. Using 1137 LUAD-specific records (out of 7192 total).
✅ Loaded 1137 slide labels from the table.

🧹 Found 1139 files left behind. Cleaning up...

✅ Moved C3L-02219-22.svs ➡️ TUMOR
✅ Moved C3L-02219-26.svs ➡️ NORMAL
✅ Moved C3L-02345-21.svs ➡️ TUMOR
✅ Moved C3L-02345-22.svs ➡️ TUMOR
✅ Moved C3L-02345-23.svs ➡️ TUMOR
✅ Moved C3L-02345-26.svs ➡️ NORMAL
✅ Moved C3L-02348-21.svs ➡️ TUMOR
✅ Moved C3L-02348-22.svs ➡️ TUMOR
✅ Moved C3L-02348-26.svs ➡️ NORMAL
✅ Moved C3L-02350-22.svs ➡️ TUMOR
✅ Moved C3L-02350-26.svs ➡️ NORMAL
✅ Moved C3L-02365-21.svs ➡️ TUMOR
✅ Moved C3L-02365-22.svs ➡️ NORMAL
✅ Moved C3L-02365-23.svs ➡️ NORMAL
✅ Moved C3L-02365-24.svs ➡️ TUMOR
✅ Moved C3L-02365-25.svs ➡️ TUMOR
✅ Moved C3L-02508-21.svs ➡️ TUMOR
✅ Moved C3L-02508-22.svs ➡️ TUMOR
✅ Moved C3L-02508-23.svs ➡️ TUMOR
✅ Moved C3L-02508-24.svs ➡️ NORMAL
✅ Moved C3L-02508-25.svs ➡️ NORMAL
✅ Moved C3L-02508-26.svs ➡️ NORMAL
✅ Moved C3L-02513-21.svs ➡️ TUMOR
✅ Moved C

In [ ]:
# Missing labels: when "Percent_Tumor_Nuclei" is "N/A" its always NORMAL and when its a number its TUMOR

import os
import glob
import pandas as pd
import shutil

# 1. Define Paths
source_folder = '/content/drive/MyDrive/CPTAC-BRCA/LUAD'
output_base_folder = '/content/drive/MyDrive/CPTAC-LUAD'

tumor_folder = '/content/drive/MyDrive/CPTAC-LUAD/TUMOR'
normal_folder = '/content/drive/MyDrive/CPTAC-LUAD/NORMAL'

csv_path = '/content/drive/MyDrive/CPTAC-LUAD/CPTAC_Data.csv'

print("Loading TCIA Master Table...")
try:
    # 2. Read the CSV
    df = pd.read_csv(csv_path)

    # Clean the column to avoid spacing errors, then filter
    df['Tumor'] = df['Tumor'].astype(str).str.strip()
    df_luad = df[df['Tumor'] == 'LUAD']

    print(f"✅ Dropped other cancers. Using {len(df_luad)} LUAD-specific records (out of {len(df)} total).")

    # Clean important columns
    df_luad['Slide_ID'] = df_luad['Slide_ID'].astype(str).str.strip()
    df_luad['Specimen_Type'] = df_luad['Specimen_Type'].astype(str).str.strip()

    # Keep Percent_Tumor_Nuclei as string so we can detect N/A safely
    df_luad['Percent_Tumor_Nuclei'] = df_luad['Percent_Tumor_Nuclei'].astype(str).str.strip()

    # 3. Build label dictionary with fallback logic
    master_dict = {}

    for _, row in df_luad.iterrows():
        slide_id = row['Slide_ID']
        specimen_type = row['Specimen_Type'].lower()
        percent_tumor_nuclei = row['Percent_Tumor_Nuclei'].strip().lower()

        # Primary rule: use Specimen_Type if available
        if "tumor" in specimen_type:
            label = "tumor"
        elif "normal" in specimen_type:
            label = "normal"

        # Fallback rule: use Percent_Tumor_Nuclei
        # rule: N/A = NORMAL, number = TUMOR
        else:
            if percent_tumor_nuclei in ["n/a", "na", "nan", "", "none"]:
                label = "normal"
            else:
                label = "tumor"

        master_dict[slide_id] = label

    print(f"✅ Loaded {len(master_dict)} slide labels from the table using Specimen_Type + Percent_Tumor_Nuclei fallback.\n")

    # 4. Get all remaining .svs files
    unsorted_files = glob.glob(os.path.join(source_folder, '*.svs'))

    if not unsorted_files:
        print("🎉 No more files left to sort in the LUAD folder!")
    else:
        print(f"🧹 Found {len(unsorted_files)} files left behind. Cleaning up...\n")

        moved_tumor = 0
        moved_normal = 0
        still_unknown = 0

        for file_path in unsorted_files:
            filename = os.path.basename(file_path)
            slide_id = os.path.splitext(filename)[0].strip()

            label = master_dict.get(slide_id, "unknown")

            if label == "tumor":
                shutil.move(file_path, os.path.join(tumor_folder, filename))
                print(f"✅ Moved {filename} ➡️ TUMOR")
                moved_tumor += 1

            elif label == "normal":
                shutil.move(file_path, os.path.join(normal_folder, filename))
                print(f"✅ Moved {filename} ➡️ NORMAL")
                moved_normal += 1

            else:
                print(f"⚠️ Unknown in CSV: {filename}")
                still_unknown += 1

        print("\n🎉 CLEANUP COMPLETE!")
        print(f"Tumor: {moved_tumor}")
        print(f"Normal: {moved_normal}")
        print(f"Still stuck: {still_unknown}")

except FileNotFoundError as e:
    print(f"❌ File or folder not found: {e}")

except KeyError as e:
    print(f"❌ Error: Could not find the expected column {e} in your CSV.")
    print("Check that the CSV has: Slide_ID, Tumor, Specimen_Type, Percent_Tumor_Nuclei")

Loading TCIA Master Table...
✅ Dropped other cancers. Using 1137 LUAD-specific records (out of 7192 total).
✅ Loaded 1137 slide labels from the table using Specimen_Type + Percent_Tumor_Nuclei fallback.

🧹 Found 76 files left behind. Cleaning up...

✅ Moved C3L-04370-21.svs ➡️ TUMOR
✅ Moved C3L-04370-22.svs ➡️ TUMOR
✅ Moved C3L-04370-23.svs ➡️ TUMOR
✅ Moved C3L-04370-24.svs ➡️ TUMOR
✅ Moved C3L-04370-25.svs ➡️ TUMOR
✅ Moved C3L-04370-26.svs ➡️ NORMAL
✅ Moved C3L-04370-27.svs ➡️ NORMAL
✅ Moved C3L-04370-28.svs ➡️ NORMAL
✅ Moved C3L-04373-21.svs ➡️ TUMOR
✅ Moved C3L-04373-22.svs ➡️ TUMOR


/tmp/ipykernel_45035/4158549416.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_luad['Slide_ID'] = df_luad['Slide_ID'].astype(str).str.strip()
/tmp/ipykernel_45035/4158549416.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_luad['Specimen_Type'] = df_luad['Specimen_Type'].astype(str).str.strip()
/tmp/ipykernel_45035/4158549416.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

✅ Moved C3L-04373-23.svs ➡️ TUMOR
✅ Moved C3L-04373-24.svs ➡️ TUMOR
✅ Moved C3L-04373-25.svs ➡️ TUMOR
✅ Moved C3L-04373-26.svs ➡️ NORMAL
✅ Moved C3L-04373-27.svs ➡️ NORMAL
✅ Moved C3L-04373-28.svs ➡️ NORMAL
✅ Moved C3L-04376-21.svs ➡️ TUMOR
✅ Moved C3L-04376-22.svs ➡️ TUMOR
✅ Moved C3L-04376-23.svs ➡️ TUMOR
✅ Moved C3L-04376-24.svs ➡️ TUMOR
✅ Moved C3L-04376-25.svs ➡️ TUMOR
✅ Moved C3L-04376-26.svs ➡️ NORMAL
✅ Moved C3L-04376-27.svs ➡️ NORMAL
✅ Moved C3L-04376-28.svs ➡️ NORMAL
✅ Moved C3L-04743-21.svs ➡️ TUMOR
✅ Moved C3L-04743-22.svs ➡️ TUMOR
✅ Moved C3L-04743-23.svs ➡️ TUMOR
✅ Moved C3L-04743-24.svs ➡️ TUMOR
✅ Moved C3L-04743-25.svs ➡️ TUMOR
✅ Moved C3L-04743-26.svs ➡️ NORMAL
✅ Moved C3L-04743-27.svs ➡️ NORMAL
✅ Moved C3L-04743-28.svs ➡️ NORMAL
✅ Moved C3L-04744-21.svs ➡️ TUMOR
✅ Moved C3L-04744-22.svs ➡️ TUMOR
✅ Moved C3L-04744-23.svs ➡️ TUMOR
✅ Moved C3L-04744-24.svs ➡️ TUMOR
✅ Moved C3L-04744-25.svs ➡️ TUMOR
✅ Moved C3L-04744-26.svs ➡️ NORMAL
✅ Moved C3L-04744-27.svs ➡️ NORMAL
✅ M

In [ ]:
# Balance split : 407 slides each

import os
import glob
import random
import shutil

# Paths
base_folder = "/content/drive/MyDrive/CPTAC-LUAD"
tumor_folder = os.path.join(base_folder, "TUMOR")
normal_folder = os.path.join(base_folder, "NORMAL")
unused_tumor_folder = os.path.join(base_folder, "UNUSED_TUMOR")

# Create unused folder
os.makedirs(unused_tumor_folder, exist_ok=True)

# Get files sorted by filename
tumor_files = sorted(glob.glob(os.path.join(tumor_folder, "*.svs")))
normal_files = sorted(glob.glob(os.path.join(normal_folder, "*.svs")))

num_tumor = len(tumor_files)
num_normal = len(normal_files)

print(f"Current tumor slides: {num_tumor}")
print(f"Current normal slides: {num_normal}")

# Target balance = number of normal slides
target_count = num_normal

if num_tumor <= target_count:
    print("✅ Tumor count is already balanced or smaller than normal count. Nothing to move.")
else:
    # Keep the first target_count tumor files, move the rest
    tumor_files_to_keep = tumor_files[:target_count]
    tumor_files_to_move = tumor_files[target_count:]

    print(f"Keeping first {len(tumor_files_to_keep)} tumor slides.")
    print(f"Moving extra {len(tumor_files_to_move)} tumor slides to UNUSED/TUMOR.")

    moved = 0

    for file_path in tumor_files_to_move:
        filename = os.path.basename(file_path)
        destination = os.path.join(unused_tumor_folder, filename)

        shutil.move(file_path, destination)
        moved += 1

    # Final counts
    final_tumor = len(glob.glob(os.path.join(tumor_folder, "*.svs")))
    final_normal = len(glob.glob(os.path.join(normal_folder, "*.svs")))
    unused_tumor = len(glob.glob(os.path.join(unused_tumor_folder, "*.svs")))

    print("\n🎉 BALANCING COMPLETE!")
    print(f"Tumor left: {final_tumor}")
    print(f"Normal left: {final_normal}")
    print(f"Unused tumor moved: {unused_tumor}")

Current tumor slides: 730
Current normal slides: 407
Keeping first 407 tumor slides.
Moving extra 323 tumor slides to UNUSED/TUMOR.

🎉 BALANCING COMPLETE!
Tumor left: 407
Normal left: 407
Unused tumor moved: 323


In [ ]:
# Rebalance 402 each (romove slides with no magnification)

import os
import glob
import shutil

# Paths
base_folder = "/content/drive/MyDrive/CPTAC-LUAD"
tumor_folder = os.path.join(base_folder, "TUMOR")
normal_folder = os.path.join(base_folder, "NORMAL")
unused_tumor_folder = os.path.join(base_folder, "UNUSED_TUMOR")

# Slides to delete
normal_missing_metadata = [
    "C3L-02513-24.svs",
    "C3L-02513-25.svs",
    "C3L-02515-24.svs",
    "C3L-02515-25.svs",
    "C3L-03642-22.svs",
]

tumor_missing_metadata = [
    "C3L-02513-21.svs",
    "C3L-02513-22.svs",
    "C3L-02513-23.svs",
    "C3L-02515-21.svs",
    "C3L-02515-22.svs",
    "C3L-02515-23.svs",
    "C3L-03642-21.svs",
]

# 1. Delete normal slides
deleted_normal = []
missing_normal = []

for filename in normal_missing_metadata:
    path = os.path.join(normal_folder, filename)

    if os.path.exists(path):
        os.remove(path)
        deleted_normal.append(filename)
    else:
        missing_normal.append(filename)

# 2. Delete tumor slides
deleted_tumor = []
missing_tumor = []

for filename in tumor_missing_metadata:
    path = os.path.join(tumor_folder, filename)

    if os.path.exists(path):
        os.remove(path)
        deleted_tumor.append(filename)
    else:
        missing_tumor.append(filename)

# 3. Move 2 tumor slides from UNUSED_TUMOR back to TUMOR
unused_files = sorted(glob.glob(os.path.join(unused_tumor_folder, "*.svs")))

files_to_restore = unused_files[:2]
restored_files = []

for file_path in files_to_restore:
    filename = os.path.basename(file_path)
    destination = os.path.join(tumor_folder, filename)

    shutil.move(file_path, destination)
    restored_files.append(filename)

# 4. Final counts
final_tumor_count = len(glob.glob(os.path.join(tumor_folder, "*.svs")))
final_normal_count = len(glob.glob(os.path.join(normal_folder, "*.svs")))
unused_tumor_count = len(glob.glob(os.path.join(unused_tumor_folder, "*.svs")))

print("✅ Cleanup complete.\n")

print(f"Deleted NORMAL slides: {len(deleted_normal)}")
for name in deleted_normal:
    print(f"  - {name}")

if missing_normal:
    print(f"\n⚠️ NORMAL slides not found: {len(missing_normal)}")
    for name in missing_normal:
        print(f"  - {name}")

print(f"\nDeleted TUMOR slides: {len(deleted_tumor)}")
for name in deleted_tumor:
    print(f"  - {name}")

if missing_tumor:
    print(f"\n⚠️ TUMOR slides not found: {len(missing_tumor)}")
    for name in missing_tumor:
        print(f"  - {name}")

print(f"\nRestored 2 tumor slides from UNUSED_TUMOR:")
for name in restored_files:
    print(f"  - {name}")

print("\n📊 Final counts:")
print(f"TUMOR: {final_tumor_count}")
print(f"NORMAL: {final_normal_count}")
print(f"UNUSED_TUMOR remaining: {unused_tumor_count}")

✅ Cleanup complete.

Deleted NORMAL slides: 5
  - C3L-02513-24.svs
  - C3L-02513-25.svs
  - C3L-02515-24.svs
  - C3L-02515-25.svs
  - C3L-03642-22.svs

Deleted TUMOR slides: 7
  - C3L-02513-21.svs
  - C3L-02513-22.svs
  - C3L-02513-23.svs
  - C3L-02515-21.svs
  - C3L-02515-22.svs
  - C3L-02515-23.svs
  - C3L-03642-21.svs

Restored 2 tumor slides from UNUSED_TUMOR:
  - C3N-01409-21.svs
  - C3N-01409-22.svs

📊 Final counts:
TUMOR: 402
NORMAL: 402
UNUSED_TUMOR remaining: 321
